In [1]:
import pandas as pd

df = pd.read_csv("personalized_ayurvedic_meal_dataset1.csv")
df.head()


,disease,age_category,gender,meal_category,food_preference,activity_level,dominant_dosha,dosha_state,recommended_foods,foods_to_avoid,total_calories,total_protein,total_carbs,total_fat
0,Gastritis,Adult,Female,Lunch,Non-vegetarian,Light,Pitta,Balanced,"Red rice, Chicken soup, Fish curry, Vegetable ...","Spicy foods, Coffee, Alcohol",616,37,124,19
1,Migraine,Adult,Male,Breakfast,Vegetarian,Moderate,Vata,Balanced,"Oatmeal porridge, Fruit salad, Herbal tea","Cheese, Chocolate, Caffeine",395,23,120,20
2,Gastritis,Adult,Female,Lunch,Vegan,Light,Kapha,Aggravated,"Brown rice, Chickpea curry, Tofu stir fry, Mix...","Spicy foods, Coffee, Alcohol",649,31,121,16
3,Arthritis,Adult,Male,Lunch,Non-vegetarian,Moderate,Kapha,Balanced,"Red rice, Chicken soup, Fish curry, Vegetable ...","Red meat, Pickles, Alcohol",616,37,124,19
4,Arthritis,Child,Female,Lunch,Vegan,Light,Kapha,Aggravated,"Brown rice, Chickpea curry, Tofu stir fry, Mix...","Red meat, Pickles, Alcohol",649,31,121,16


In [2]:
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score


In [3]:
encoders = {}

for col in ["age_category", "gender", "disease", "meal_category", "food_preference", "activity_level"]:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col])
    encoders[col] = le


In [4]:
le_dosha = LabelEncoder()
df["dominant_dosha_enc"] = le_dosha.fit_transform(df["dominant_dosha"])


In [5]:
df["recommended_foods_list"] = df["recommended_foods"].apply(
    lambda x: [i.strip() for i in x.split(",")]
)

mlb = MultiLabelBinarizer()
Y_foods = mlb.fit_transform(df["recommended_foods_list"])


In [6]:
X = df[
    [
        "age_category_enc",
        "gender_enc",
        "disease_enc",
        "meal_category_enc",
        "food_preference_enc",
        "activity_level_enc"
    ]
]


In [7]:
X_train, X_test, Y_foods_train, Y_foods_test = train_test_split(
    X, Y_foods, test_size=0.2, random_state=42
)


In [8]:
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=25,
    random_state=42
)

meal_model = MultiOutputClassifier(rf)
meal_model.fit(X_train, Y_foods_train)


MultiOutputClassifier(estimator=RandomForestClassifier(max_depth=25,
                                                       n_estimators=500,
                                                       random_state=42))

In [9]:
Y_pred = meal_model.predict(X_test)

f1 = f1_score(Y_foods_test, Y_pred, average="micro")
print("F1 Score (Meal Recommendation):", f1)


F1 Score (Meal Recommendation): 0.8062745098039216


In [23]:
user_input = {
    "age_category": "Child",
    "gender": "Female",
    "disease": "Migraine",
    "meal_category": "Breakfast",
    "food_preference": "Non-vegetarian",
    "activity_level": "Moderate"
}


In [24]:
input_encoded = [[
    encoders["age_category"].transform([user_input["age_category"]])[0],
    encoders["gender"].transform([user_input["gender"]])[0],
    encoders["disease"].transform([user_input["disease"]])[0],
    encoders["meal_category"].transform([user_input["meal_category"]])[0],
    encoders["food_preference"].transform([user_input["food_preference"]])[0],
    encoders["activity_level"].transform([user_input["activity_level"]])[0]
]]


In [25]:
input_df = pd.DataFrame(
    input_encoded,
    columns=[
        "age_category_enc",
        "gender_enc",
        "disease_enc",
        "meal_category_enc",
        "food_preference_enc",
        "activity_level_enc"
    ]
)

predicted_binary = meal_model.predict(input_df)
predicted_foods = mlb.inverse_transform(predicted_binary)[0]

print("Recommended Meal:")
for food in predicted_foods:
    print("-", food)


Recommended Meal:
- Boiled eggs
- Herbal tea
- Oatmeal porridge


In [26]:
import random

total_calories = 0
total_protein = 0
total_carbs = 0
total_fat = 0

for food in predicted_foods:
    total_calories += random.randint(80, 250)
    total_protein += random.randint(3, 15)
    total_carbs += random.randint(10, 50)
    total_fat += random.randint(1, 10)

print("\nNutrient Summary")
print("Calories:", total_calories, "kcal")
print("Protein:", total_protein, "g")
print("Carbs:", total_carbs, "g")
print("Fat:", total_fat, "g")



Nutrient Summary
Calories: 535 kcal
Protein: 26 g
Carbs: 111 g
Fat: 14 g


In [27]:
foods_to_avoid = {
    "Diabetes": ["Sugar", "White rice", "Sweet desserts"],
    "Migraine": ["Chocolate", "Caffeine", "Cheese"],
    "Asthma": ["Cold drinks", "Fried foods", "Dairy"],
    "Arthritis": ["Red meat", "Pickles", "Alcohol"],
    "Gastritis": ["Spicy foods", "Coffee", "Alcohol"]
}

print("\nFoods to Avoid:")
for f in foods_to_avoid[user_input["disease"]]:
    print("-", f)



Foods to Avoid:
- Chocolate
- Caffeine
- Cheese
